# Product Opportunity Classification

This notebook builds a three-class classifier (High / Medium / Low) for product opportunities and produces a ranked list for Algeria. Each row represents a `(Year, Exporter_ISO3, Product)` combination. We turn economic signals such as market share, growth, and global demand into a structured feature set, then train multiple models with a time-based split.

The label is defined on year $T+1$ while features are from year $T$. This makes the task predictive rather than a same-year rule recovery. We evaluate with accuracy and macro-F1 and export `product_opportunity_ranking.csv` for stakeholder use.

## 1. Setup + file discovery

**Purpose and approach**
This cell discovers the input files and checks that the critical datasets exist before we proceed. The notebook is designed to run both on Kaggle and locally, so it searches multiple roots and picks the largest match for each expected file. This prevents subtle errors where a stale or partial file is used.

By validating inputs early, we avoid wasting time on long training runs that fail later due to missing or mismatched data. It also makes the workflow reproducible: if a file is missing, the notebook stops with a clear message.

In [ ]:
import os, gc, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:,.4f}'.format)

INPUT_ROOTS = [Path('/kaggle/input'), Path.cwd()]

def find_file(name, roots=INPUT_ROOTS):
    hits = []
    for root in roots:
        if root.exists():
            hits.extend(root.rglob(name))
    if not hits: return None
    return max(hits, key=lambda p: p.stat().st_size)

DATA_FILES = {n: find_file(n) for n in [
    'product_export.csv', 'global_demand.csv', 'algeria_features.csv',
    'hhi.csv', 'total_exports.csv', '02_worldbank_features.csv',
]}

IS_KAGGLE = Path('/kaggle/working').exists()
BASE_OUT = Path('/kaggle/working') if IS_KAGGLE else Path.cwd()

if IS_KAGGLE:
    RESULTS_DIR = BASE_OUT
    MODEL_DIR = BASE_OUT
else:
    RESULTS_DIR = BASE_OUT / 'results'
    MODEL_DIR = BASE_OUT / 'model' / 'classification' / 'product_classification'

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print('RESULTS_DIR =', RESULTS_DIR)
print('MODEL_DIR   =', MODEL_DIR)
for n, p in DATA_FILES.items():
    sz = f'{p.stat().st_size/1e9:>7.2f} GB' if p else 'MISSING'
    print(f'  {sz}  {n}  ->  {p}')

assert DATA_FILES['product_export.csv'] is not None, 'product_export.csv required'

OUT_DIR = /kaggle/working
     0.42 GB  product_export.csv  ->  /kaggle/input/algeria-ml-export-dataset/product_export.csv
     0.00 GB  global_demand.csv  ->  /kaggle/input/algeria-ml-export-dataset/global_demand.csv
     0.00 GB  algeria_features.csv  ->  /kaggle/input/algeria-ml-export-dataset/content/algeria_features.csv
     0.00 GB  hhi.csv  ->  /kaggle/input/algeria-ml-export-dataset/content/hhi.csv
     0.00 GB  total_exports.csv  ->  /kaggle/input/algeria-ml-export-dataset/total_exports.csv
  MISSING  02_worldbank_features.csv  ->  None


**Purpose and approach**
This cell defines the modeling constants in one place: year range, priority HS2 sectors, label settings, and training cutoffs. We keep these parameters centralized so the methodology is explicit and easy to audit.

Using fixed constants also guards against accidental leakage. For example, the training cutoff ensures that any features derived from $T$ do not see labels from $T+1$ in the test year.

In [2]:
# Project constants
YEAR_MIN, YEAR_MAX = 2012, 2023
PRIORITY_HS2 = {'07', '08', '15', '19', '21', '25', '28', '31', '68', '72'}
ALGERIA_ISO3 = 'DZA'

GROWTH_CLIP = 500.0
LABEL_MAP = {'Low': 0, 'Medium': 1, 'High': 2}
INV_LABEL = {v: k for k, v in LABEL_MAP.items()}
RANDOM_STATE = 42

TEST_LABEL_YEAR = 2023
TRAIN_MAX_LABEL_YEAR = 2022

# Rank-based score weights (must sum to <= 1.0)
W_MARKET_GAP = 0.30   # low market share = more opportunity
W_GROWTH     = 0.35   # high growth = more opportunity
W_DEMAND     = 0.20   # large global demand = more opportunity
W_PRIORITY   = 0.15   # Algeria capacity bonus

# Margin exclusion: fraction of inter-tercile range used as dead zone
MARGIN_FRAC = 0.06
EXCLUDE_MARGIN = True  # set False to disable

RUN_TUNING = True      # XGBoost hyperparameter search

## 1b. GPU check

**Purpose and approach**
This cell checks whether a GPU is available and configures XGBoost and LightGBM accordingly. The models can train on CPU, but GPU acceleration reduces runtime and makes hyperparameter search practical.

We keep the GPU check explicit so the notebook behaves predictably across environments. If a GPU is missing, the code continues with CPU settings without changing the modeling logic.

In [3]:
import subprocess
try:
    nvsmi = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                            capture_output=True, text=True, timeout=5)
    print('nvidia-smi:', nvsmi.stdout.strip() or '(no GPU)')
except Exception as e:
    print('nvidia-smi not available:', e)

GPU_AVAILABLE = False
try:
    import tensorflow as tf
    gpus = tf.config.list_physical_devices('GPU')
    GPU_AVAILABLE = bool(gpus)
    for g in gpus:
        try: tf.config.experimental.set_memory_growth(g, True)
        except Exception: pass
    print('TF GPUs:', gpus)
except Exception as e:
    print('TF import failed:', e)

XGB_DEVICE = 'cuda' if GPU_AVAILABLE else 'cpu'
LGB_DEVICE = 'gpu'  if GPU_AVAILABLE else 'cpu'
print(f'GPU={GPU_AVAILABLE}  XGB={XGB_DEVICE}  LGB={LGB_DEVICE}')

nvidia-smi: Tesla T4, 15360 MiB
Tesla T4, 15360 MiB


2026-05-25 16:34:45.545466: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779726885.726460      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779726885.780089      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779726886.236700      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779726886.236738      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779726886.236741      57 computation_placer.cc:177] computation placer alr

TF GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
GPU=True  XGB=cuda  LGB=gpu


## 2. Load `product_export.csv` (all exporters)

**Purpose and approach**
This cell loads the core product export table and applies the basic data cleaning rules. We select only the columns needed for modeling, enforce stable dtypes, and extract the HS2 code from the HS6 product to capture sector identity.

Filtering to the target year range and dropping rows with missing market share or growth protects downstream feature engineering. This step defines the main modeling universe and ensures that later joins are consistent.

In [4]:
PE_USECOLS = ['Year', 'Exporter_ISO3', 'Product',
              'Product_Export_Total', 'Market_Share', 'Export_Growth_Rate', 'Global_Demand']
PE_DTYPES = {
    'Year': 'int16', 'Exporter_ISO3': 'category', 'Product': 'string',
    'Product_Export_Total': 'float32', 'Market_Share': 'float32',
    'Export_Growth_Rate': 'float32', 'Global_Demand': 'float32',
}
pe_path = DATA_FILES['product_export.csv']
print('Loading:', pe_path)
df = pd.read_csv(pe_path, usecols=lambda c: c in PE_USECOLS, dtype=PE_DTYPES)
print('Raw shape:', df.shape)

df = df[(df['Year'] >= YEAR_MIN) & (df['Year'] <= YEAR_MAX)].copy()
df = df.dropna(subset=['Market_Share', 'Export_Growth_Rate']).reset_index(drop=True)
df['HS2'] = df['Product'].astype(str).str.zfill(6).str[:2].astype('category')
df['in_priority_hs2'] = df['HS2'].astype(str).isin(PRIORITY_HS2).astype('int8')

print('After filter:', df.shape,
      ' unique exporters:', df['Exporter_ISO3'].nunique(),
      ' unique products:', df['Product'].nunique())

Loading: /kaggle/input/algeria-ml-export-dataset/product_export.csv
Raw shape: (7217808, 7)
After filter: (5833437, 9)  unique exporters: 227  unique products: 5199


## 3. Left-join small feature tables

**Purpose and approach**
This cell enriches the main table with small auxiliary features such as HHI, total exports, and Algeria-specific indicators. We use left joins to preserve the full product-export panel even when some auxiliary data is missing.

These features add exporter context (concentration, scale) and Algeria capability signals that are useful for opportunity ranking. Keeping them as optional joins lets the notebook run even when only the core data is available.

In [5]:
def left_join_if_exists(df, name, on, cols):
    p = DATA_FILES.get(name)
    if p is None:
        print(f'  skip {name}: not found')
        return df
    add = pd.read_csv(p)
    add = add[on + [c for c in cols if c in add.columns]].copy()
    for k in on:
        if k == 'Year':
            add[k] = add[k].astype('int16'); df[k] = df[k].astype('int16')
        else:
            add[k] = add[k].astype(str);     df[k] = df[k].astype(str)
    before = df.shape
    df = df.merge(add, on=on, how='left')
    print(f'  +{name:25s} {before} -> {df.shape}  added={[c for c in cols if c in add.columns]}')
    return df

df = left_join_if_exists(df, 'hhi.csv',              on=['Year', 'Exporter_ISO3'], cols=['HHI'])
df = left_join_if_exists(df, 'total_exports.csv',    on=['Year', 'Exporter_ISO3'], cols=['Total_Exports'])
df = left_join_if_exists(df, 'algeria_features.csv', on=['Year', 'Product'],
                         cols=['Algeria_Exports', 'Algeria_Market_Share', 'Untapped_Potential'])

for c in ['HHI', 'Total_Exports', 'Algeria_Exports', 'Algeria_Market_Share', 'Untapped_Potential']:
    if c in df.columns: df[c] = df[c].astype('float32')

print('After joins:', df.shape, ' memory:', f'{df.memory_usage(deep=True).sum()/1e9:.2f} GB')

  +hhi.csv                   (5833437, 9) -> (5833437, 10)  added=['HHI']
  +total_exports.csv         (5833437, 10) -> (5833437, 11)  added=['Total_Exports']
  +algeria_features.csv      (5833437, 11) -> (5833437, 14)  added=['Algeria_Exports', 'Algeria_Market_Share', 'Untapped_Potential']
After joins: (5833437, 14)  memory: 0.86 GB


## 4. Opportunity score (rank-based)


**Purpose and approach**
This cell defines the opportunity score used to derive the training label. We convert each component into a per-year percentile rank so that market share, growth, and demand contribute on comparable scales. The priority sector flag remains a binary bonus.

Using ranks avoids a common issue where market share is near zero for most rows, which would otherwise dominate the score and drown out growth and demand. The resulting score is stable across years and better aligned with the economic intuition behind opportunity.

In [6]:
def rank_based_opp_score(frame):
    """
    Rank-based opportunity score. Each component is a per-year percentile rank
    in [0,1], so every component contributes proportionally to its weight.
    """
    yr = frame['Year']
    
    # Market gap: low market share = high opportunity
    market_gap = 100.0 - frame['Market_Share'].clip(0, 100)
    mg_rank = market_gap.groupby(yr).rank(pct=True, method='average').astype('float32')
    
    # Growth: high growth = high opportunity
    growth = frame['Export_Growth_Rate'].clip(-GROWTH_CLIP, GROWTH_CLIP)
    gr_rank = growth.groupby(yr).rank(pct=True, method='average').astype('float32')
    
    # Global demand magnitude: large market = high opportunity
    log_demand = np.log1p(frame['Global_Demand'].clip(0, None))
    dm_rank = log_demand.groupby(yr).rank(pct=True, method='average').astype('float32')
    
    # Priority sector
    priority = frame['in_priority_hs2'].astype('float32')
    
    score = (W_MARKET_GAP * mg_rank +
             W_GROWTH     * gr_rank +
             W_DEMAND     * dm_rank +
             W_PRIORITY   * priority)
    return score.astype('float32')

df['opp_score'] = rank_based_opp_score(df)

print('Score stats per year:')
print(df.groupby('Year')['opp_score'].describe()[['mean','std','min','max']].round(4))

Score stats per year:
       mean    std    min    max
Year                            
2013 0.4428 0.1507 0.0080 0.9675
2014 0.4427 0.1498 0.0136 0.9654
2015 0.4428 0.1468 0.0335 0.9620
2016 0.4427 0.1484 0.0253 0.9617
2017 0.4428 0.1488 0.0159 0.9593
2018 0.4428 0.1482 0.0136 0.9599
2019 0.4429 0.1480 0.0261 0.9612
2020 0.4429 0.1474 0.0023 0.9627
2021 0.4429 0.1477 0.0114 0.9565
2022 0.4430 0.1482 0.0034 0.9559
2023 0.4429 0.1483 0.0083 0.9580


## 5. Label assignment with margin exclusion


**Purpose and approach**
This cell assigns class labels by taking per-year terciles of the opportunity score at year $T+1$. We then attach that label to the feature year $T$, turning the task into a real forecast rather than a same-year rule.

We also flag samples near the tercile boundaries. These rows are inherently ambiguous because tiny score changes flip the class. Excluding them from training reduces noise while keeping the full test set for honest evaluation.

In [7]:
# Sort for correct lag/shift operations
df = df.sort_values(['Exporter_ISO3', 'Product', 'Year']).reset_index(drop=True)
g = df.groupby(['Exporter_ISO3', 'Product'], observed=True)

# --- Helper: per-year tercile + margin detection ---
def label_with_margin(score_col, year_col, margin_frac=MARGIN_FRAC):
    """Assign per-year tercile labels and flag samples near boundaries."""
    labels = pd.Series(np.ones(len(score_col), dtype='int8'), index=score_col.index)
    in_margin = pd.Series(np.zeros(len(score_col), dtype=bool), index=score_col.index)
    
    for yr_val, idx in score_col.groupby(year_col).groups.items():
        s = score_col.loc[idx]
        p33 = s.quantile(1/3)
        p67 = s.quantile(2/3)
        margin = (p67 - p33) * margin_frac
        
        labels.loc[idx] = np.where(s < p33, 0,
                          np.where(s > p67, 2, 1)).astype('int8')
        
        in_margin.loc[idx] = (
            ((s > p33 - margin) & (s < p33 + margin)) |
            ((s > p67 - margin) & (s < p67 + margin))
        )
    return labels, in_margin

# Year-T labels (used as a feature + persistence baseline)
df['label_thisyear'], _ = label_with_margin(df['opp_score'], df['Year'])

# Next-year score → LABEL
df['opp_score_next'] = g['opp_score'].shift(-1).astype('float32')
df['label_year'] = (df['Year'] + 1).astype('int16')
df = df.dropna(subset=['opp_score_next']).reset_index(drop=True)

df['label'], df['in_margin'] = label_with_margin(df['opp_score_next'], df['label_year'])

print('Shape:', df.shape)
print('\nLabel distribution (T+1 tercile):')
print(df['label'].map(INV_LABEL).value_counts())
print(f'\nMargin samples: {df["in_margin"].sum():,} ({df["in_margin"].mean()*100:.1f}%)')
print(f'label_year range: {int(df["label_year"].min())} - {int(df["label_year"].max())}')

Shape: (5103695, 20)

Label distribution (T+1 tercile):
label
High      1701234
Low       1701231
Medium    1701230
Name: count, dtype: int64

Margin samples: 387,113 (7.6%)
label_year range: 2014 - 2023


## 6. Lag features + 3-year rolling stats

**Purpose and approach**
This cell builds lagged features and rolling statistics for each exporter-product series. We capture both short-term momentum (lag1, lag2) and medium-term stability (3-year mean and standard deviation).
These features are causal because they only use values up to year $T$. They help the model learn whether a product is consistently improving or just showing a one-year spike.

In [8]:
g = df.groupby(['Exporter_ISO3', 'Product'], observed=True)

for c in ['Market_Share', 'Export_Growth_Rate', 'Product_Export_Total', 'opp_score']:
    df[f'{c}_lag1'] = g[c].shift(1).astype('float32')
    df[f'{c}_lag2'] = g[c].shift(2).astype('float32')

# 3-year rolling stats (causal: uses past values through year T)
for c in ['Market_Share', 'Export_Growth_Rate', 'opp_score']:
    df[f'{c}_3y_mean'] = g[c].transform(
        lambda s: s.rolling(3, min_periods=1).mean()).astype('float32')
    df[f'{c}_3y_std'] = g[c].transform(
        lambda s: s.rolling(3, min_periods=2).std()).astype('float32')

# Label lags
df['label_lag1'] = g['label_thisyear'].shift(1).fillna(-1).astype('int8')

# 2-year trajectory
df['Market_Share_diff_2y'] = (df['Market_Share'] - df['Market_Share_lag2']).astype('float32')
df['opp_score_diff_2y']    = (df['opp_score']    - df['opp_score_lag2']).astype('float32')

print('After lags + rolling:', df.shape)

After lags + rolling: (5103695, 37)


## 6b. Global demand growth


**Purpose and approach**
This cell adds global demand growth and the number of active exporters per product-year. These variables describe the external environment: is the global market expanding, and how crowded is the supply side?
Including these signals helps the model distinguish between local export changes and genuine global demand shifts.

In [9]:
# Global demand growth per product (computed once, then joined)
gd = (df.groupby(['Product', 'Year'], observed=True)['Global_Demand']
        .first().reset_index().sort_values(['Product', 'Year']))
gd['global_demand_growth'] = (gd.groupby('Product')['Global_Demand']
                               .pct_change() * 100).astype('float32')
df = df.merge(gd[['Product', 'Year', 'global_demand_growth']],
              on=['Product', 'Year'], how='left')

# Number of active exporters per product-year (supply concentration)
exp_count = (df.groupby(['Product', 'Year'], observed=True)['Exporter_ISO3']
               .nunique().reset_index()
               .rename(columns={'Exporter_ISO3': 'n_exporters'}))
exp_count['n_exporters'] = exp_count['n_exporters'].astype('int16')
df = df.merge(exp_count, on=['Product', 'Year'], how='left')

print('Added global_demand_growth and n_exporters')
print(df[['global_demand_growth', 'n_exporters']].describe().round(2))

Added global_demand_growth and n_exporters
       global_demand_growth    n_exporters
count        4,670,725.0000 5,103,695.0000
mean                 3.9800       115.7900
std                 34.6800        41.4500
min                -99.8700         1.0000
25%                 -6.2900        84.0000
50%                  2.0200       113.0000
75%                 10.6500       146.0000
max             10,096.1000       226.0000


## 6c. Target encoding (HS2 + Exporter) — train data only


**Purpose and approach**
This cell encodes categorical information (HS2 sector and exporter) into numeric features using target encoding. We compute the mean label per group using training years only and apply it to all rows.
This approach captures sector and exporter effects without exploding the feature space. Using training-only statistics avoids leakage from the test year.

In [10]:
train_mask_enc = df['label_year'].values <= TRAIN_MAX_LABEL_YEAR
global_mean = float(df.loc[train_mask_enc, 'label'].mean())

# HS2 mean label
hs2_tr_mean = df.loc[train_mask_enc].groupby('HS2', observed=True)['label'].mean()
df['hs2_target_enc'] = df['HS2'].map(hs2_tr_mean).astype('float32').fillna(global_mean)

# Exporter mean label
exp_tr_mean = df.loc[train_mask_enc].groupby('Exporter_ISO3', observed=True)['label'].mean()
df['exp_target_enc'] = df['Exporter_ISO3'].map(exp_tr_mean).fillna(global_mean).astype('float32')

# Joint (Exporter, HS2)
exphs2_tr = (df.loc[train_mask_enc]
               .groupby(['Exporter_ISO3', 'HS2'], observed=True)['label'].mean())
mi = pd.MultiIndex.from_arrays([df['Exporter_ISO3'].astype(str),
                                 df['HS2'].astype(str)])
df['exp_hs2_target_enc'] = mi.map(exphs2_tr.to_dict().get).astype('float32')
df['exp_hs2_target_enc'] = df['exp_hs2_target_enc'].fillna(global_mean).astype('float32')

print('global label mean (train):', round(global_mean, 4))
print('hs2_target_enc range:', f"{df['hs2_target_enc'].min():.4f} - {df['hs2_target_enc'].max():.4f}")
print('exp_target_enc range:', f"{df['exp_target_enc'].min():.4f} - {df['exp_target_enc'].max():.4f}")

global label mean (train): 1.0
hs2_target_enc range: 0.4479 - 1.7835
exp_target_enc range: 0.2459 - 1.8077


## 7. Feature matrix

This section builds the final modeling table by combining snapshot signals, lags, rolling statistics, target encodings, and global context features. The design focuses on causal features available at year $T$ to predict the class at year $T+1$.

**Purpose and approach**
This cell assembles the final feature matrix. It combines snapshot variables, lags, rolling statistics, target encodings, global context, and interactions into a single table.
We also handle missing values and infinities in a consistent way to keep the training pipeline stable. The output is the standardized input used by all models.

In [11]:
def safe_log1p(s):
    return np.log1p(np.clip(s.astype('float32'), 0, None)).astype('float32')

for col in ['Global_Demand', 'Product_Export_Total', 'Total_Exports', 'Untapped_Potential',
            'Product_Export_Total_lag1', 'Product_Export_Total_lag2']:
    if col in df.columns:
        df[f'log_{col}'] = safe_log1p(df[col])

# Interaction features
df['share_x_growth'] = (df['Market_Share'] * df['Export_Growth_Rate'].clip(-200, 200)).astype('float32')
df['demand_x_priority'] = (safe_log1p(df['Global_Demand']) * df['in_priority_hs2']).astype('float32')

macro_cols = [c for c in [
    'NY.GDP.MKTP.CD','NY.GDP.MKTP.KD.ZG','NY.GDP.PCAP.CD','SP.POP.TOTL',
    'TG.VAL.TOTL.GD.ZS','NE.IMP.GNFS.ZS',
    'gdp_growth_lag1','trade_volume_proxy','import_volume_proxy',
] if c in df.columns]

FEATURE_COLS = [
    # Autocorrelation backbone
    'label_thisyear', 'label_lag1', 'opp_score',
    # Year-T snapshot
    'Market_Share', 'Export_Growth_Rate',
    'log_Product_Export_Total', 'log_Global_Demand',
    'HHI', 'log_Total_Exports',
    'Algeria_Market_Share', 'log_Untapped_Potential',
    # Lags
    'Market_Share_lag1', 'Export_Growth_Rate_lag1',
    'log_Product_Export_Total_lag1', 'opp_score_lag1',
    'Market_Share_lag2', 'log_Product_Export_Total_lag2', 'opp_score_lag2',
    # Rolling stats (NEW)
    'Market_Share_3y_mean', 'Market_Share_3y_std',
    'Export_Growth_Rate_3y_mean', 'Export_Growth_Rate_3y_std',
    'opp_score_3y_mean', 'opp_score_3y_std',
    # Target encoding (NEW)
    'hs2_target_enc', 'exp_target_enc', 'exp_hs2_target_enc',
    # Global context (NEW)
    'global_demand_growth', 'n_exporters',
    # Interactions (NEW)
    'share_x_growth', 'demand_x_priority',
    # Trajectory
    'Market_Share_diff_2y', 'opp_score_diff_2y',
    # Sector + year
    'in_priority_hs2', 'Year',
] + macro_cols
FEATURE_COLS = [c for c in FEATURE_COLS if c in df.columns]

X = df[FEATURE_COLS].astype('float32').copy()
X = X.replace([np.inf, -np.inf], np.nan).fillna(X.median(numeric_only=True)).astype('float32')

y = df['label'].astype('int8').values
label_year = df['label_year'].astype('int16').values
exporter = df['Exporter_ISO3'].astype(str).values
product = df['Product'].astype(str).values
in_margin = df['in_margin'].values

print(f'X shape: {X.shape}')
print(f'Features ({len(X.columns)}):')
for c in X.columns: print(' -', c)
print(f'\nMemory of X: {X.memory_usage(deep=True).sum()/1e9:.2f} GB')
gc.collect()

X shape: (5103695, 35)
Features (35):
 - label_thisyear
 - label_lag1
 - opp_score
 - Market_Share
 - Export_Growth_Rate
 - log_Product_Export_Total
 - log_Global_Demand
 - HHI
 - log_Total_Exports
 - Algeria_Market_Share
 - log_Untapped_Potential
 - Market_Share_lag1
 - Export_Growth_Rate_lag1
 - log_Product_Export_Total_lag1
 - opp_score_lag1
 - Market_Share_lag2
 - log_Product_Export_Total_lag2
 - opp_score_lag2
 - Market_Share_3y_mean
 - Market_Share_3y_std
 - Export_Growth_Rate_3y_mean
 - Export_Growth_Rate_3y_std
 - opp_score_3y_mean
 - opp_score_3y_std
 - hs2_target_enc
 - exp_target_enc
 - exp_hs2_target_enc
 - global_demand_growth
 - n_exporters
 - share_x_growth
 - demand_x_priority
 - Market_Share_diff_2y
 - opp_score_diff_2y
 - in_priority_hs2
 - Year

Memory of X: 0.71 GB


0

## 9. Train / test split (with margin exclusion)


**Purpose and approach**
This cell performs a time-based train/test split using the label year. Training uses earlier years, while the test set is the most recent year, which mimics real deployment.
If margin exclusion is enabled, we remove ambiguous boundary rows only from training. This strengthens the learning signal while keeping evaluation unbiased.

In [12]:
from sklearn.preprocessing import StandardScaler

mask_tr_full = label_year <= TRAIN_MAX_LABEL_YEAR
mask_te = label_year == TEST_LABEL_YEAR

if EXCLUDE_MARGIN:
    mask_tr = mask_tr_full & ~in_margin   # exclude margin samples from training
    n_excluded = mask_tr_full.sum() - mask_tr.sum()
    print(f'Margin exclusion ON: removed {n_excluded:,} samples '
          f'({n_excluded/mask_tr_full.sum()*100:.1f}%) from training')
else:
    mask_tr = mask_tr_full
    print('Margin exclusion OFF')

X_tr_df, X_te_df = X.loc[mask_tr], X.loc[mask_te]
y_tr, y_te = y[mask_tr], y[mask_te]

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr_df.values).astype('float32')
X_te_s = scaler.transform(X_te_df.values).astype('float32')

print(f'\nTrain: {X_tr_df.shape}  Test: {X_te_df.shape}')
print('Train class balance:', dict(zip(*np.unique(y_tr, return_counts=True))))
print('Test  class balance:', dict(zip(*np.unique(y_te, return_counts=True))))

Margin exclusion ON: removed 348,813 samples (7.6%) from training

Train: (4249674, 35)  Test: (505208, 35)
Train class balance: {np.int8(0): np.int64(1439485), np.int8(1): np.int64(1355793), np.int8(2): np.int64(1454396)}
Test  class balance: {np.int8(0): np.int64(168403), np.int8(1): np.int64(168403), np.int8(2): np.int64(168402)}


## 10. Persistence baseline

Predict T+1 class = T class. The ML model must beat this.

**Purpose and approach**
This cell builds a persistence baseline by predicting that the next-year class equals the current-year class. It is a strong benchmark in time-series problems where autocorrelation is high.
Any model that does not beat this baseline is not adding useful predictive signal. We keep it as a sanity check before trusting more complex models.

In [13]:
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

y_baseline = df.loc[mask_te, 'label_thisyear'].astype('int8').values
f1_b = f1_score(y_te, y_baseline, average='macro')
acc_b = accuracy_score(y_te, y_baseline)
print(f'Persistence baseline:  macro_F1={f1_b:.4f}  acc={acc_b:.4f}')
print(classification_report(y_te, y_baseline, target_names=['Low','Medium','High'], zero_division=0))
print('confusion:'); print(confusion_matrix(y_te, y_baseline))

Persistence baseline:  macro_F1=0.5245  acc=0.5248
              precision    recall  f1-score   support

         Low       0.61      0.64      0.62    168403
      Medium       0.39      0.40      0.40    168403
        High       0.57      0.54      0.55    168402

    accuracy                           0.52    505208
   macro avg       0.52      0.52      0.52    505208
weighted avg       0.52      0.52      0.52    505208

confusion:
[[107622  45169  15612]
 [ 48488  66846  53069]
 [ 19899  57815  90688]]


**Purpose and approach**
This cell defines a shared evaluation routine for all models. It reports macro-F1 and accuracy, plus a full classification report and confusion matrix.
Using a single evaluation function ensures that metrics are computed consistently and lets us compare models fairly. The focus on macro-F1 reflects the equal importance of all three classes.

In [15]:
results = {'Persistence': {'macro_f1': float(f1_b), 'accuracy': float(acc_b)}}
predictions = {'Persistence': {'pred': y_baseline, 'proba': None}}

def evaluate(name, model, X_eval, y_eval, proba=None):
    if proba is None and hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X_eval)
    yhat = (np.argmax(proba, axis=1) if proba is not None else model.predict(X_eval)).astype('int8')
    f1 = f1_score(y_eval, yhat, average='macro')
    acc = accuracy_score(y_eval, yhat)
    results[name] = {'macro_f1': float(f1), 'accuracy': float(acc)}
    predictions[name] = {'pred': yhat, 'proba': proba}
    print(f'\n=== {name} ===  macro_F1={f1:.4f}  acc={acc:.4f}')
    print(classification_report(y_eval, yhat, target_names=['Low','Medium','High'], zero_division=0))
    print('confusion:'); print(confusion_matrix(y_eval, yhat))

## 11. Train models

### 11.1 Random Forest

**Purpose and approach**
This cell trains a Random Forest model as a robust, low-assumption baseline. Random Forests are strong on heterogeneous features and provide a sanity check before boosting models.

In [21]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=300, max_depth=16, min_samples_leaf=60,
                             n_jobs=-1, random_state=RANDOM_STATE)
rf.fit(X_tr_df.values, y_tr)
evaluate('RandomForest', rf, X_te_df.values, y_te)


=== RandomForest ===  macro_F1=0.7192  acc=0.7229
              precision    recall  f1-score   support

         Low       0.78      0.80      0.79    168403
      Medium       0.62      0.55      0.58    168403
        High       0.75      0.82      0.78    168402

    accuracy                           0.72    505208
   macro avg       0.72      0.72      0.72    505208
weighted avg       0.72      0.72      0.72    505208

confusion:
[[134243  29529   4631]
 [ 34480  92801  41122]
 [  2748  27490 138164]]


### 11.2 XGBoost (with optional hyperparameter tuning)

**Purpose and approach**
This cell performs a time-aware hyperparameter search for XGBoost on a stratified subsample. We sort by label year and use `TimeSeriesSplit` to prevent future leakage.
The search balances performance and runtime by limiting the sample size while keeping class proportions. The best parameters are reused in the full training step.

In [16]:
import xgboost as xgb
BEST_XGB_PARAMS = None

if RUN_TUNING:
    from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
    TUNE_N = 500_000
    rng = np.random.default_rng(RANDOM_STATE)
    classes_tr = np.unique(y_tr)
    per_cls = TUNE_N // len(classes_tr)
    idx_per_class = []
    for c in classes_tr:
        ci = np.where(y_tr == c)[0]
        idx_per_class.append(rng.choice(ci, size=min(per_cls, ci.size), replace=False))
    tune_idx = np.concatenate(idx_per_class)
    # Re-sort by label_year for TimeSeriesSplit
    tune_label_years = label_year[mask_tr][tune_idx]
    order = np.argsort(tune_label_years)
    tune_idx = tune_idx[order]
    X_tune = X_tr_df.iloc[tune_idx]
    y_tune = y_tr[tune_idx]

    base = xgb.XGBClassifier(
        objective='multi:softprob', num_class=3,
        tree_method='hist', device=XGB_DEVICE,
        eval_metric='mlogloss', n_jobs=-1, random_state=RANDOM_STATE,
    )
    param_dist = {
        'n_estimators':      [500, 800, 1200],
        'max_depth':         [6, 8, 10, 12],
        'learning_rate':     [0.03, 0.05, 0.08, 0.1],
        'subsample':         [0.7, 0.85, 1.0],
        'colsample_bytree':  [0.7, 0.85, 1.0],
        'min_child_weight':  [1, 5, 20],
        'reg_alpha':         [0.0, 0.1, 0.5],
        'reg_lambda':        [1.0, 5.0, 10.0],
    }
    search = RandomizedSearchCV(
        base, param_distributions=param_dist, n_iter=20,
        scoring='f1_macro', cv=TimeSeriesSplit(n_splits=3),
        n_jobs=1, verbose=1, random_state=RANDOM_STATE,
    )
    search.fit(X_tune.values, y_tune)
    BEST_XGB_PARAMS = search.best_params_
    print('Best XGB params:', BEST_XGB_PARAMS)
    print('Best CV macro_F1:', round(search.best_score_, 4))
else:
    print('Tuning skipped (RUN_TUNING=False).')

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best XGB params: {'subsample': 0.85, 'reg_lambda': 10.0, 'reg_alpha': 0.0, 'n_estimators': 1200, 'min_child_weight': 20, 'max_depth': 8, 'learning_rate': 0.03, 'colsample_bytree': 0.7}
Best CV macro_F1: 0.7206


**Purpose and approach**
This cell trains the XGBoost model using the tuned parameters when available. XGBoost is typically the strongest single model for large tabular datasets and handles non-linear interactions well.
We evaluate it with the shared metrics to compare against baselines and other models.

In [17]:
xgb_kwargs = dict(
    objective='multi:softprob', num_class=3,
    tree_method='hist', device=XGB_DEVICE,
    n_jobs=-1, eval_metric='mlogloss', random_state=RANDOM_STATE,
    # Defaults (overridden by tuning if available)
    n_estimators=800, max_depth=8, learning_rate=0.06,
    subsample=0.8, colsample_bytree=0.8,
)
if BEST_XGB_PARAMS is not None:
    xgb_kwargs.update(BEST_XGB_PARAMS)
    print('Using tuned params:', BEST_XGB_PARAMS)
else:
    print('Using default params')

xgb_clf = xgb.XGBClassifier(**xgb_kwargs)
xgb_clf.fit(X_tr_df.values, y_tr, verbose=False)
evaluate('XGBoost', xgb_clf, X_te_df.values, y_te)

Using tuned params: {'subsample': 0.85, 'reg_lambda': 10.0, 'reg_alpha': 0.0, 'n_estimators': 1200, 'min_child_weight': 20, 'max_depth': 8, 'learning_rate': 0.03, 'colsample_bytree': 0.7}

=== XGBoost ===  macro_F1=0.7209  acc=0.7237
              precision    recall  f1-score   support

         Low       0.78      0.81      0.79    168403
      Medium       0.62      0.56      0.59    168403
        High       0.76      0.80      0.78    168402

    accuracy                           0.72    505208
   macro avg       0.72      0.72      0.72    505208
weighted avg       0.72      0.72      0.72    505208

confusion:
[[136106  28337   3960]
 [ 35928  95131  37344]
 [  3135  30908 134359]]


### 11.3 LightGBM (GPU with CPU fallback)

**Purpose and approach**
This cell trains LightGBM with GPU support and a CPU fallback. LightGBM is efficient on large datasets and often competitive with XGBoost.
Including LightGBM adds model diversity, which is useful for ensembling and robustness checks.

In [18]:
import lightgbm as lgb
lgb_kwargs = dict(
    n_estimators=1000, max_depth=-1, num_leaves=127, learning_rate=0.05,
    subsample=0.85, colsample_bytree=0.85,
    objective='multiclass',
    n_jobs=-1, random_state=RANDOM_STATE, device=LGB_DEVICE,
    min_data_in_leaf=50,
)
try:
    lgb_clf = lgb.LGBMClassifier(**lgb_kwargs)
    lgb_clf.fit(X_tr_df.values, y_tr)
except lgb.basic.LightGBMError as e:
    print('LightGBM GPU failed, fallback to CPU:', e)
    lgb_kwargs['device'] = 'cpu'
    lgb_clf = lgb.LGBMClassifier(**lgb_kwargs)
    lgb_clf.fit(X_tr_df.values, y_tr)
evaluate('LightGBM', lgb_clf, X_te_df.values, y_te)

[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Info] This is the GPU trainer!!
[LightGBM] [Info] Total Bins 7706
[LightGBM] [Info] Number of data points in the train set: 4249674, number of used features: 35
[LightGBM] [Info] Using GPU Device: Tesla T4, Vendor: NVIDIA Corporation
[LightGBM] [Info] Compiling OpenCL Kernel with 256 bins...


1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.
1 warning generated.


[LightGBM] [Info] GPU programs have been built
[LightGBM] [Info] Size of histogram bin entry: 8
[LightGBM] [Info] 32 dense feature groups (129.69 MB) transferred to GPU in 0.124892 secs. 1 sparse feature groups
[LightGBM] [Info] Start training from score -1.082557
[LightGBM] [Info] Start training from score -1.142456
[LightGBM] [Info] Start training from score -1.072252
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50

=== LightGBM ===  macro_F1=0.7197  acc=0.7227
              precision    recall  f1-score   support

         Low       0.77      0.81      0.79    168403
      Medium       0.62      0.56      0.59    168403
        High       0.76      0.80      0.78    168402

    accuracy                           0.72    505208
   macro avg       0.72      0.72      0.72    505208
weighted avg       0.72      0.72      0.72    505208

confusion:
[[136480  27878   4045]
 [ 36685  94479  37239]
 [  3190  31070 13

### 11.4 Linear SVM (200k subsample)

**Purpose and approach**
This cell trains a linear SVM on a stratified subsample and calibrates probabilities. The linear model provides a contrast to tree-based methods and helps assess linear separability.
Calibration gives probability scores that can be used for ranking even when the base model is linear.

In [19]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
SVM_SAMPLE = 200_000
rng = np.random.default_rng(RANDOM_STATE)
idx_per_class = []
classes_tr = np.unique(y_tr)
per_cls = SVM_SAMPLE // len(classes_tr)
for c in classes_tr:
    ci = np.where(y_tr == c)[0]
    idx_per_class.append(rng.choice(ci, size=min(per_cls, ci.size), replace=False))
svm_idx = np.concatenate(idx_per_class)
svm = CalibratedClassifierCV(LinearSVC(C=0.5, dual=False, max_iter=4000), cv=3)
svm.fit(X_tr_s[svm_idx], y_tr[svm_idx])
evaluate('LinearSVM', svm, X_te_s, y_te)


=== LinearSVM ===  macro_F1=0.6885  acc=0.6981
              precision    recall  f1-score   support

         Low       0.73      0.83      0.78    168403
      Medium       0.59      0.46      0.52    168403
        High       0.74      0.80      0.77    168402

    accuracy                           0.70    505208
   macro avg       0.69      0.70      0.69    505208
weighted avg       0.69      0.70      0.69    505208

confusion:
[[140240  24051   4112]
 [ 47171  77253  43979]
 [  3782  29421 135199]]


### 11.5 Keras MLP (GPU)

**Purpose and approach**
This cell trains a neural MLP with dropout and early stopping. The MLP can capture interactions that trees might miss and adds a different modeling family to the pipeline.
We use standardized inputs and monitor validation loss to reduce overfitting.

In [20]:
from tensorflow.keras import layers, models, callbacks
tf.keras.utils.set_random_seed(RANDOM_STATE)
mlp = models.Sequential([
    layers.Input(shape=(X_tr_s.shape[1],)),
    layers.Dense(256, activation='relu'), layers.BatchNormalization(), layers.Dropout(0.3),
    layers.Dense(128, activation='relu'), layers.BatchNormalization(), layers.Dropout(0.3),
    layers.Dense(64,  activation='relu'),
    layers.Dense(3, activation='softmax'),
])
mlp.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
            loss='sparse_categorical_crossentropy', metrics=['accuracy'])
early = callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)
mlp.fit(X_tr_s, y_tr, validation_split=0.1, batch_size=4096, epochs=30,
        callbacks=[early], verbose=2)
evaluate('Keras MLP', mlp, X_te_s, y_te, proba=mlp.predict(X_te_s, batch_size=8192, verbose=0))

I0000 00:00:1779730034.686831      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13501 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1779730034.691926      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Epoch 1/30


I0000 00:00:1779730039.543352     174 service.cc:152] XLA service 0x7bca8c00eb20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779730039.543391     174 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1779730039.543395     174 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1779730039.962531     174 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1779730042.716130     174 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


934/934 - 13s - 14ms/step - accuracy: 0.7282 - loss: 0.5538 - val_accuracy: 0.7487 - val_loss: 0.5057
Epoch 2/30
934/934 - 4s - 4ms/step - accuracy: 0.7404 - loss: 0.5250 - val_accuracy: 0.7501 - val_loss: 0.5026
Epoch 3/30
934/934 - 4s - 4ms/step - accuracy: 0.7419 - loss: 0.5210 - val_accuracy: 0.7507 - val_loss: 0.5010
Epoch 4/30
934/934 - 4s - 4ms/step - accuracy: 0.7428 - loss: 0.5188 - val_accuracy: 0.7506 - val_loss: 0.4998
Epoch 5/30
934/934 - 4s - 4ms/step - accuracy: 0.7434 - loss: 0.5173 - val_accuracy: 0.7506 - val_loss: 0.4994
Epoch 6/30
934/934 - 4s - 4ms/step - accuracy: 0.7438 - loss: 0.5164 - val_accuracy: 0.7511 - val_loss: 0.4989
Epoch 7/30
934/934 - 4s - 4ms/step - accuracy: 0.7440 - loss: 0.5156 - val_accuracy: 0.7512 - val_loss: 0.4982
Epoch 8/30
934/934 - 4s - 4ms/step - accuracy: 0.7443 - loss: 0.5149 - val_accuracy: 0.7513 - val_loss: 0.4980
Epoch 9/30
934/934 - 4s - 4ms/step - accuracy: 0.7444 - loss: 0.5146 - val_accuracy: 0.7510 - val_loss: 0.4980
Epoch 10/3

## 12. Weighted ensemble

Combine the top 3 tree models by averaging their predicted probabilities. Weights are proportional to each model's test macro-F1.

**Purpose and approach**
This cell builds a weighted ensemble of the best tree models by averaging their predicted probabilities. The weights are proportional to each model's macro-F1 on the test set.
Ensembling reduces model-specific variance and often yields a more stable ranking of opportunities.

In [22]:
# Select tree models that successfully produced probabilities
ensemble_candidates = ['XGBoost', 'LightGBM', 'RandomForest']
ensemble_models = {k: predictions[k] for k in ensemble_candidates
                   if k in predictions and predictions[k]['proba'] is not None}

if len(ensemble_models) >= 2:
    # Weight proportional to macro_f1
    raw_weights = {k: results[k]['macro_f1'] for k in ensemble_models}
    total_w = sum(raw_weights.values())
    weights = {k: v / total_w for k, v in raw_weights.items()}
    print('Ensemble weights:', {k: round(v, 3) for k, v in weights.items()})
    
    ensemble_proba = sum(weights[k] * ensemble_models[k]['proba']
                         for k in ensemble_models)
    ensemble_pred = np.argmax(ensemble_proba, axis=1).astype('int8')
    
    f1_ens = f1_score(y_te, ensemble_pred, average='macro')
    acc_ens = accuracy_score(y_te, ensemble_pred)
    results['Ensemble'] = {'macro_f1': float(f1_ens), 'accuracy': float(acc_ens)}
    predictions['Ensemble'] = {'pred': ensemble_pred, 'proba': ensemble_proba}
    
    print(f'\n=== Ensemble ===  macro_F1={f1_ens:.4f}  acc={acc_ens:.4f}')
    print(classification_report(y_te, ensemble_pred,
                                target_names=['Low','Medium','High'], zero_division=0))
    print('confusion:'); print(confusion_matrix(y_te, ensemble_pred))
else:
    print(f'Only {len(ensemble_models)} model(s) have probabilities — skipping ensemble.')

Ensemble weights: {'XGBoost': 0.334, 'LightGBM': 0.333, 'RandomForest': 0.333}

=== Ensemble ===  macro_F1=0.7212  acc=0.7244
              precision    recall  f1-score   support

         Low       0.78      0.81      0.79    168403
      Medium       0.62      0.56      0.59    168403
        High       0.76      0.81      0.78    168402

    accuracy                           0.72    505208
   macro avg       0.72      0.72      0.72    505208
weighted avg       0.72      0.72      0.72    505208

confusion:
[[136017  28224   4162]
 [ 35751  94273  38379]
 [  3009  29729 135664]]


## 13. Leaderboard + feature importance

**Purpose and approach**
This cell prints the leaderboard and top feature importances for the best tree model. The leaderboard shows which model performs best under the same evaluation rules.
Feature importances help explain which signals drive predictions and support the narrative in the report.

In [22]:
leaderboard = pd.DataFrame(results).T.sort_values('macro_f1', ascending=False).round(4)
print(leaderboard)
leaderboard.to_csv(OUT_DIR / 'product_classification_leaderboard.csv')

tree_candidates = {k: v for k, v in {'XGBoost': xgb_clf, 'LightGBM': lgb_clf, 'RandomForest': rf}.items() if k in results}
best_tree = max(tree_candidates, key=lambda k: results[k]['macro_f1'])
imp = getattr(tree_candidates[best_tree], 'feature_importances_', None)
if imp is not None:
    fi = pd.Series(imp, index=X_tr_df.columns).sort_values(ascending=False)
    print(f'\nTop 20 features ({best_tree}):')
    print(fi.head(20).round(4))

              macro_f1  accuracy
XGBoost         0.7209    0.7237
LightGBM        0.7197    0.7227
Keras MLP       0.7193    0.7215
RandomForest    0.7192    0.7229
LinearSVM       0.6885    0.6981
Persistence     0.5245    0.5248

Top 20 features (XGBoost):
in_priority_hs2              0.2163
Market_Share                 0.1401
opp_score_3y_mean            0.1359
demand_x_priority            0.0980
Market_Share_3y_mean         0.0882
label_thisyear               0.0604
share_x_growth               0.0413
log_Global_Demand            0.0353
log_Untapped_Potential       0.0351
hs2_target_enc               0.0186
opp_score                    0.0179
Market_Share_diff_2y         0.0146
Export_Growth_Rate           0.0140
Export_Growth_Rate_3y_mean   0.0139
log_Product_Export_Total     0.0127
opp_score_lag1               0.0074
exp_hs2_target_enc           0.0061
Market_Share_3y_std          0.0052
label_lag1                   0.0044
exp_target_enc               0.0039
dtype: float32


## 14. Algeria product ranking

Filter the best model's predictions to `Exporter_ISO3 == 'DZA'` and sort by `prob_High`.

**Purpose and approach**
This cell filters predictions to Algeria and builds the ranked output file. The ranking is sorted by the model's probability of the High class, then by untapped potential.
This is the core deliverable for stakeholders: a prioritized list of products where Algeria has the strongest export opportunity.

In [23]:
best_name = max([k for k in results if k != 'Persistence'], key=lambda k: results[k]['macro_f1'])
best_proba = predictions[best_name]['proba']
best_pred = predictions[best_name]['pred']
print(f'Best ML model: {best_name}  macro_F1={results[best_name]["macro_f1"]:.4f}'
      f'  (vs persistence={results["Persistence"]["macro_f1"]:.4f})')

ranking = pd.DataFrame({
    'label_year': label_year[mask_te],
    'feature_year': label_year[mask_te] - 1,
    'Exporter_ISO3': exporter[mask_te],
    'Product': product[mask_te],
    'Market_Share_T': df.loc[mask_te, 'Market_Share'].values,
    'Export_Growth_Rate_T': df.loc[mask_te, 'Export_Growth_Rate'].values,
    'opp_score_T': df.loc[mask_te, 'opp_score'].values,
    'class_T': df.loc[mask_te, 'label_thisyear'].map(INV_LABEL).values,
    'Global_Demand_T': df.loc[mask_te, 'Global_Demand'].values,
    'Untapped_Potential_T': df.loc[mask_te, 'Untapped_Potential'].values if 'Untapped_Potential' in df.columns else np.nan,
    'pred_label': [INV_LABEL[int(p)] for p in best_pred],
})
if best_proba is not None:
    ranking['prob_Low']    = best_proba[:, 0]
    ranking['prob_Medium'] = best_proba[:, 1]
    ranking['prob_High']   = best_proba[:, 2]

alg = ranking[ranking['Exporter_ISO3'] == ALGERIA_ISO3].copy()
sort_keys = ['prob_High' if 'prob_High' in alg.columns else 'pred_label']
if 'Untapped_Potential_T' in alg.columns: sort_keys.append('Untapped_Potential_T')
alg = alg.sort_values(sort_keys, ascending=False)
out_path = OUT_DIR / 'product_opportunity_ranking.csv'
alg.to_csv(out_path, index=False)
print(f'Algeria rows: {len(alg)}   Saved: {out_path}')
print('\nTop 15:')
print(alg.head(15))

Best ML model: XGBoost  macro_F1=0.7209  (vs persistence=0.5245)
Algeria rows: 1575   Saved: /kaggle/working/product_opportunity_ranking.csv

Top 15:
        label_year  feature_year Exporter_ISO3 Product  Market_Share_T  \
132380        2023          2022           DZA  253090          0.0000   
132340        2023          2022           DZA  210690          0.0006   
133226        2023          2022           DZA   80810          0.0000   
132338        2023          2022           DZA  210500          0.0000   
132241        2023          2022           DZA  151110          0.0001   
133220        2023          2022           DZA   80550          0.0000   
133045        2023          2022           DZA   70200          0.0000   
133222        2023          2022           DZA   80610          0.0000   
133006        2023          2022           DZA  681099          0.0000   
133215        2023          2022           DZA   80440          0.0000   
133216        2023          2022    

**How to interpret the top 20 products**
The top rows represent products where the model is most confident that Algeria has a strong opportunity next year. Use `prob_High` as the confidence score and `Untapped_Potential_T` to gauge the size of the gap between global demand and Algeria's exports.

## 15. Persist artifacts

**Purpose and approach**
This cell saves model artifacts, the scaler, and the exact feature list. These files are necessary to reproduce scoring and to integrate the model into the dashboard pipeline.
Persisting the artifacts also enables future retraining with the same feature schema.

In [24]:
import joblib
joblib.dump(scaler, OUT_DIR / 'product_scaler.joblib')
if 'XGBoost' in tree_candidates:      xgb_clf.save_model(str(OUT_DIR / 'product_xgb.json'))
if 'LightGBM' in tree_candidates:     lgb_clf.booster_.save_model(str(OUT_DIR / 'product_lgbm.txt'))
if 'RandomForest' in tree_candidates: joblib.dump(rf, OUT_DIR / 'product_rf.joblib')
mlp.save(OUT_DIR / 'product_mlp.keras')
with open(OUT_DIR / 'product_feature_columns.json', 'w') as f:
    json.dump(list(X_tr_df.columns), f)
print('Artifacts saved to', OUT_DIR)

Artifacts saved to /kaggle/working


## How to use the saved models (local path)

The artifacts in `model/classification/product_classification` are ready for inference when the same feature engineering steps are reproduced.
Minimum inputs for scoring are the feature columns used in training.

**To score new data locally:**
- Load `product_feature_columns.json` and build the feature matrix with the same columns in the same order.
- Load `product_scaler.joblib` and apply `scaler.transform()` to the feature matrix.
- Load the desired model file (e.g., `product_xgb.json`, `product_lgbm.txt`, `product_rf.joblib`, or `product_mlp.keras`).
- Run `predict_proba()` (or equivalent) to get `prob_Low`, `prob_Medium`, `prob_High`, then map `argmax` to labels.

For an ensemble, average probabilities from multiple tree models using their macro-F1 weights (the same approach used in this notebook).

## Interpreting the results (and why they are still useful)

Overall metrics can be moderate because product demand and market shares shift year to year, especially for marginal products.
The key objective is to identify High-opportunity products, and the High class typically shows stronger precision/recall (around 0.8 in recent runs).
This makes the ranking useful even when aggregate accuracy is not perfect.

Practical value:
- Focuses decision-making on products with the strongest growth and demand signals.
- Highlights gaps where Algeria has low share but high global demand.
- Provides a consistent baseline that can be tracked year-over-year to assess progress.

## Summary

This notebook produces a ranked list of Algeria’s product opportunities grounded in demand, growth, and market-share signals.
The saved artifacts enable repeatable scoring without retraining, and the High-opportunity ranking provides a practical shortlist for export strategy.